# 3.10 Pandas for Data Analysis

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 3: Python for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-03-python-for-machine-learning/03.10_pandas-for-data-analysis.ipynb)

Companion code for Section 3.10 of the book. Run the cells in order from top to bottom.

**Example 3.10.1**

In [ ]:
import pandas as pd
import numpy as np

# ── Series - 1D labelled array ─────────────────────────────────────
# Like a dictionary with a numeric index
accuracy_by_model = pd.Series(
    [0.87, 0.94, 0.97, 0.93],
    index=['LogReg', 'XGBoost', 'ResNet50', 'LightGBM'],
    name='test_accuracy'
)
print(accuracy_by_model)
print(f'Best model: {accuracy_by_model.idxmax()} ({accuracy_by_model.max():.2%})')
print(f'Mean:  {accuracy_by_model.mean():.4f}')
print(f'Above 0.93: {accuracy_by_model[accuracy_by_model > 0.93].index.tolist()}')

# ── DataFrame - 2D labelled table ─────────────────────────────────
# Method 1: from dict (most common for small data)
df = pd.DataFrame({
    'model':    ['LogReg', 'XGBoost', 'ResNet50', 'LightGBM', 'SVM'],
    'accuracy': [0.87, 0.94, 0.97, 0.93, 0.89],
    'f1_score': [0.865, 0.938, 0.971, 0.925, 0.882],
    'train_s':  [0.5, 12.3, 450.0, 5.8, 8.1],
    'params_k': [105, 890, 25_600, 720, 0.1],
})

print('\nDataFrame:')
print(df)
print(f'\nShape: {df.shape}')       # (5, 5)
print(f'Columns: {list(df.columns)}')
print(f'Dtypes:\n{df.dtypes}')

**Example 3.10.2**

In [ ]:
import pandas as pd
import numpy as np

# Load the Titanic dataset (classic ML dataset)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
# For offline: df = pd.read_csv('titanic.csv')
# We'll simulate it:
np.random.seed(42)
n = 891
df = pd.DataFrame({
    'PassengerId': range(1, n+1),
    'Survived':    np.random.choice([0,1], n, p=[0.62,0.38]),
    'Pclass':      np.random.choice([1,2,3], n, p=[0.24,0.21,0.55]),
    'Name':        [f'Person_{i}' for i in range(n)],
    'Sex':         np.random.choice(['male','female'], n, p=[0.65,0.35]),
    'Age':         np.where(np.random.rand(n)<0.2, np.nan,
                   np.random.normal(29, 14, n).clip(1,80).round(1)),
    'SibSp':       np.random.choice([0,1,2,3], n, p=[0.68,0.23,0.06,0.03]),
    'Fare':        np.random.exponential(32, n).round(2),
    'Embarked':    np.random.choice(['S','C','Q',None], n, p=[0.72,0.19,0.08,0.01]),
})

# ── Essential inspection methods ──────────────────────────────────
print('=== df.head(5) - first 5 rows ===')
print(df.head(5))

print('\n=== df.info() - column types and nulls ===')
df.info()

print('\n=== df.describe() - summary statistics ===')
print(df.describe().round(3))

print('\n=== Missing values ===')
print(df.isnull().sum())
print(f'Total missing: {df.isnull().sum().sum()}')
print(f'Missing %: {df.isnull().mean().mul(100).round(1)}')

print('\n=== Value counts ===')
print(df['Pclass'].value_counts())
print(df['Sex'].value_counts(normalize=True).mul(100).round(1))

**Example 3.10.3**

In [ ]:
# ── Handling Missing Values ──────────────────────────────────────
print(f'Before cleaning: {df.isnull().sum().sum()} missing values')

# Fill numeric missing with median (robust to outliers)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill categorical missing with mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop columns with too many missing values (>50% threshold)
missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.5].index.tolist()
df.drop(columns=cols_to_drop, inplace=True)

print(f'After cleaning:  {df.isnull().sum().sum()} missing values')

# ── Feature Engineering ───────────────────────────────────────────
# Create new features from existing ones
df['FamilySize'] = df['SibSp'] + 1   # 1 = alone, >4 = large family
df['IsAlone']    = (df['SibSp'] == 0).astype(int)
df['Fare_log']   = np.log1p(df['Fare'])  # Log-transform skewed feature

# Age binning - discretise continuous to categorical
df['AgeGroup'] = pd.cut(df['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child','Teen','Adult','Middle','Senior']
)

# ── Encoding Categorical Variables ───────────────────────────────
# One-hot encoding: Sex, Embarked
df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked', 'AgeGroup'],
                             drop_first=True, dtype=int)
print(f'\nShape after encoding: {df_encoded.shape}')
print('New columns:', [c for c in df_encoded.columns if '_' in c][:8])

# Label encoding for ordinal features
pclass_map = {1: 2, 2: 1, 3: 0}   # Higher class → higher number
df['Pclass_enc'] = df['Pclass'].map(pclass_map)

# ── Handling Outliers ────────────────────────────────────────────
Q1   = df['Fare'].quantile(0.25)
Q3   = df['Fare'].quantile(0.75)
IQR  = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

n_outliers = ((df['Fare'] < lower) | (df['Fare'] > upper)).sum()
print(f'\nFare outliers: {n_outliers}')

# Cap outliers (winsorisation) instead of removing
df['Fare_capped'] = df['Fare'].clip(lower=lower, upper=upper)
print(f'Fare range before capping: [{df.Fare.min():.1f}, {df.Fare.max():.1f}]')
print(f'Fare range after capping:  [{df.Fare_capped.min():.1f}, {df.Fare_capped.max():.1f}]')

**Example 3.10.4**

In [ ]:
# ── Column Selection ─────────────────────────────────────────────
# Single column → Series
ages = df['Age']                    # Series

# Multiple columns → DataFrame
subset = df[['Age','Fare','Survived']]

# Select by dtype
numeric_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols      = df.select_dtypes(include=['object','category']).columns.tolist()
print(f'Numeric columns: {numeric_cols}')

# ── Row Selection (loc and iloc) ──────────────────────────────────
# .loc: label-based indexing
row_0  = df.loc[0]                  # Row with index label 0
subset = df.loc[0:4, ['Age','Survived']]  # Rows 0-4, specific cols

# .iloc: integer-position based indexing (like NumPy)
first5    = df.iloc[:5]             # First 5 rows
last3     = df.iloc[-3:]            # Last 3 rows
cell      = df.iloc[2, 3]          # Row 2, col 3

# ── Filtering with Boolean Conditions ────────────────────────────
# Simple condition
survivors = df[df['Survived'] == 1]
first_class = df[df['Pclass'] == 1]

# Multiple conditions: use & (and), | (or), ~ (not)
wealthy_survivors = df[(df['Survived']==1) & (df['Pclass']==1)]
child_or_senior   = df[(df['Age'] < 12) | (df['Age'] > 60)]

# .query() method - often more readable
high_fare = df.query('Fare > 50 and Survived == 1')
print(f'High-fare survivors: {len(high_fare)}')

# ── Applying Functions ───────────────────────────────────────────
# .apply() - apply function to each element, row, or column
df['Fare_binned'] = df['Fare'].apply(lambda x: 'high' if x>50 else 'low')

# .map() - for element-wise transformation of a Series
df['Sex_binary'] = df['Sex'].map({'male': 0, 'female': 1})

# Vectorised string operations via .str accessor
df['Name_upper'] = df['Name'].str.upper()
df['Name_len']   = df['Name'].str.len()
has_person = df['Name'].str.contains('Person_1', na=False)

print(df[['Name','Sex','Sex_binary','Fare','Fare_binned']].head(5))

**Example 3.10.5**

In [ ]:
# Survival rate by passenger class and gender
survival_by_class = df.groupby('Pclass')['Survived'].mean()
print('Survival rate by class:')
print(survival_by_class.round(3))

# Multiple aggregations at once
summary = df.groupby('Pclass').agg(
    n_passengers = ('Survived', 'count'),
    survival_rate= ('Survived', 'mean'),
    mean_age     = ('Age', 'mean'),
    mean_fare    = ('Fare', 'mean'),
    fare_std     = ('Fare', 'std')
).round(3)
print('\nPassenger summary by class:')
print(summary)

# Groupby multiple columns
class_gender = df.groupby(['Pclass','Sex'])['Survived'].agg(['mean','count'])
class_gender.columns = ['survival_rate','count']
print('\nSurvival by class and gender:')
print(class_gender.round(3))

# Transform - broadcast group statistic back to original shape
# Useful for creating group-normalised features
df['Fare_vs_class_mean'] = df.groupby('Pclass')['Fare'].transform(
    lambda x: (x - x.mean()) / x.std()   # z-score within each class
)
print(f'\nFare z-score by class: mean={df.Fare_vs_class_mean.mean():.4f}')

# Pivot table - cross-tabulation
pivot = df.pivot_table(
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean'
).round(3)
print('\nSurvival rate (pivot table):')
print(pivot)

**Example 3.10.6**

In [ ]:
import pandas as pd
import numpy as np

# ── Merge (like SQL JOIN) ─────────────────────────────────────────
passengers = pd.DataFrame({
    'id':     [1,2,3,4,5],
    'name':   ['Alice','Bob','Charlie','Diana','Eve'],
    'age':    [25, 32, 45, 28, 35]
})

predictions = pd.DataFrame({
    'id':       [1,2,3,4,6],          # Note: id=5 missing, id=6 extra
    'prediction':[1, 0, 1, 1, 0],
    'confidence':[0.92, 0.87, 0.95, 0.78, 0.65]
})

# Inner join: only matching ids
inner = pd.merge(passengers, predictions, on='id', how='inner')
print(f'Inner join: {len(inner)} rows (ids 1-4 only)')

# Left join: all from passengers, NaN where no match
left = pd.merge(passengers, predictions, on='id', how='left')
print(f'Left join:  {len(left)} rows (all passengers, Eve has NaN prediction)')
print(left)

# ── Concatenate - stack DataFrames ───────────────────────────────
train_df  = pd.DataFrame({'X': [1,2,3], 'y': [0,1,0], 'split': 'train'})
test_df   = pd.DataFrame({'X': [4,5,6], 'y': [1,0,1], 'split': 'test'})
full_df   = pd.concat([train_df, test_df], ignore_index=True)
print(f'\nConcatenated: {full_df.shape}')

# ── Reshaping ─────────────────────────────────────────────────────
# melt: wide format → long format
wide = pd.DataFrame({
    'model': ['LogReg','XGBoost'],
    'train_acc': [0.88, 0.95],
    'val_acc':   [0.86, 0.93],
    'test_acc':  [0.85, 0.92]
})

long = wide.melt(id_vars='model', var_name='split', value_name='accuracy')
print('\nMelted (long format):')
print(long)

**Example 3.10.7**

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np

# Load dataset into DataFrame
data = load_breast_cancer()
df_bc = pd.DataFrame(data.data, columns=data.feature_names)
df_bc['target'] = data.target

print('=== Data Overview ===')
print(f'Shape: {df_bc.shape}')
print(f'Missing values: {df_bc.isnull().sum().sum()}')
print(f'Class balance: {df_bc.target.value_counts().to_dict()}')

print('\n=== Summary Statistics (first 5 features) ===')
print(df_bc.iloc[:,:5].describe().round(3))

print('\n=== Correlation with target ===')
corr = df_bc.corr()['target'].drop('target').sort_values()
print('Top 5 negative correlations (worst predictors of benign):')
print(corr.head(5).round(4))
print('Top 5 positive correlations (best predictors of benign):')
print(corr.tail(5).round(4))

# Build ML-ready feature matrix
X = df_bc.drop('target', axis=1)
y = df_bc['target']

# Remove low-variance features
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_sel = selector.fit_transform(X)
print(f'\nFeatures after variance filter: {X.shape[1]} → {X_sel.shape[1]}')

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)
print(f'Feature mean after scaling: {X_scaled.mean():.4f}  (should be 0)')
print(f'Feature std  after scaling: {X_scaled.std():.4f}   (should be 1)')